# SkyPortal corpus — reproducibility (C)

This notebook tests one thing: is `data/corpus_skyportal/` a deterministic function of
the raw captures under `data/raw/skyportal/`? It regenerates both stages
(`01_flatten.py`, `02_normalise.py`) into an isolated temporary directory and compares
the result byte-for-byte against what is committed on disk.

Unlike notebooks A and B, this notebook reads the corpus directly — that is deliberate.
Here the corpus is not a source of truth to build evidence from; it is the artefact under
test, and the raw captures plus the two scripts are the ground truth it is tested against.

In [1]:
import hashlib
import json
import re
import shutil
import subprocess
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", 250)

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "data/interim/skyportal_corpus").is_dir())
VENV_PYTHON = ROOT / ".venv/bin/python"
INTERIM_ROOT = ROOT / "data/interim/skyportal_corpus"
CORPUS_ROOT = ROOT / "data/corpus_skyportal"
TABLE_NAMES = ["sources", "comments", "photometry", "spectra", "followup_requests"]
FIELD_HISTORY_TABLE = "source_field_history"  # corpus-only: expanded, not transformed


def sha256_of_file(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()


def fingerprint(root, stage):
    rows = []
    for name in TABLE_NAMES:
        path = root / f"{name}.parquet"
        frame = pd.read_parquet(path)
        rows.append({"file": f"{stage}/{name}.parquet", "rows": len(frame),
                     "columns": frame.shape[1], "sha256": sha256_of_file(path)})
    return rows


baseline_rows = fingerprint(INTERIM_ROOT, "interim") + fingerprint(CORPUS_ROOT, "corpus")
history_path = CORPUS_ROOT / f"{FIELD_HISTORY_TABLE}.parquet"
history_frame = pd.read_parquet(history_path)
baseline_rows.append({"file": f"interim/{FIELD_HISTORY_TABLE}.parquet", "rows": "n/a",
                      "columns": "n/a", "sha256": "n/a"})  # no interim counterpart
baseline_rows.append({"file": f"corpus/{FIELD_HISTORY_TABLE}.parquet", "rows": len(history_frame),
                      "columns": history_frame.shape[1], "sha256": sha256_of_file(history_path)})
baseline = pd.DataFrame(baseline_rows)
baseline

,file,rows,columns,sha256
0,interim/sources.parquet,982,114,f50ef622a28285f750d940fc5dfb39d5f7405360c0519f63d6600bde49e49a6b
1,interim/comments.parquet,2950,13,4dc1e66febd156516d49867b20a6a1a55363c1334ba1bd3544a435974c7774b0
2,interim/photometry.parquet,7968,44,db93bd722c22d938df389b6cdf2815d4d5c58b4b38f8da54af3085df8b4baa0f
3,interim/spectra.parquet,1,52,ddb9ce9421dfa87c3990398d1aac078393d83bf7a385d6edbbd89be3290a69f3
4,interim/followup_requests.parquet,2359,164,22d695fb99e75e44e23ed69daa5f436de908e9402ebf7ea80794f2d03c38c890
5,corpus/sources.parquet,800,91,6a25a96ec4fb66581f07c9ceac9d109c67416d1cc80ad30a6ab83039e77529ad
6,corpus/comments.parquet,2950,11,523eae742a6632a07eaef273307c2f9697f47f2bdf4df7b052fb940aff9b3a54
7,corpus/photometry.parquet,7968,43,9d2919bbffcb45e9054206ed1c5cf7d568ea0f4a369e0492b642e750523f99f9
8,corpus/spectra.parquet,1,42,b55cac3b37be1af851d8a0c45e1a3fb996f851248fea61515b797fa1bd0e8fef
9,corpus/followup_requests.parquet,2339,147,942330e7d9cff1f54871106def821411393ab93f6eeda4c37755d77393806d66


In [2]:
tmp_root = Path(tempfile.mkdtemp(prefix="skyportal_repro_"))
(tmp_root / "scripts/skyportal").mkdir(parents=True)
shutil.copy2(ROOT / "scripts/skyportal/01_flatten.py", tmp_root / "scripts/skyportal/01_flatten.py")
shutil.copy2(ROOT / "scripts/skyportal/02_normalise.py", tmp_root / "scripts/skyportal/02_normalise.py")
(tmp_root / "data").mkdir()
(tmp_root / "data/raw").symlink_to(ROOT / "data/raw")  # same raw captures, not a copy

for script in ["01_flatten.py", "02_normalise.py"]:
    result = subprocess.run([str(VENV_PYTHON), str(tmp_root / "scripts/skyportal" / script)],
                            capture_output=True, text=True)
    print(f"{script}: exit={result.returncode}")
    if result.returncode != 0:
        print(result.stderr[-2000:])

regen_interim = tmp_root / "data/interim/skyportal_corpus"
regen_corpus = tmp_root / "data/corpus_skyportal"

rows = []
for stage, on_disk_root, regen_root in [("interim", INTERIM_ROOT, regen_interim),
                                        ("corpus", CORPUS_ROOT, regen_corpus)]:
    for name in TABLE_NAMES:
        on_disk_hash = sha256_of_file(on_disk_root / f"{name}.parquet")
        regen_hash = sha256_of_file(regen_root / f"{name}.parquet")
        rows.append({"table": name, "stage": stage, "sha256_on_disk": on_disk_hash,
                     "sha256_regenerated": regen_hash, "match": on_disk_hash == regen_hash})

history_on_disk = sha256_of_file(CORPUS_ROOT / f"{FIELD_HISTORY_TABLE}.parquet")
history_regen = sha256_of_file(regen_corpus / f"{FIELD_HISTORY_TABLE}.parquet")
rows.append({"table": FIELD_HISTORY_TABLE, "stage": "corpus", "sha256_on_disk": history_on_disk,
            "sha256_regenerated": history_regen, "match": history_on_disk == history_regen})

regeneration = pd.DataFrame(rows)
print(f"\nall {len(regeneration)} hashes match: {bool(regeneration['match'].all())}")
print(f"interim hashes: {int((regeneration['stage'] == 'interim').sum())} | "
     f"corpus hashes: {int((regeneration['stage'] == 'corpus').sum())}")
regeneration

01_flatten.py: exit=0


02_normalise.py: exit=0

all 11 hashes match: True
interim hashes: 5 | corpus hashes: 6


,table,stage,sha256_on_disk,sha256_regenerated,match
0,sources,interim,f50ef622a28285f750d940fc5dfb39d5f7405360c0519f63d6600bde49e49a6b,f50ef622a28285f750d940fc5dfb39d5f7405360c0519f63d6600bde49e49a6b,True
1,comments,interim,4dc1e66febd156516d49867b20a6a1a55363c1334ba1bd3544a435974c7774b0,4dc1e66febd156516d49867b20a6a1a55363c1334ba1bd3544a435974c7774b0,True
2,photometry,interim,db93bd722c22d938df389b6cdf2815d4d5c58b4b38f8da54af3085df8b4baa0f,db93bd722c22d938df389b6cdf2815d4d5c58b4b38f8da54af3085df8b4baa0f,True
3,spectra,interim,ddb9ce9421dfa87c3990398d1aac078393d83bf7a385d6edbbd89be3290a69f3,ddb9ce9421dfa87c3990398d1aac078393d83bf7a385d6edbbd89be3290a69f3,True
4,followup_requests,interim,22d695fb99e75e44e23ed69daa5f436de908e9402ebf7ea80794f2d03c38c890,22d695fb99e75e44e23ed69daa5f436de908e9402ebf7ea80794f2d03c38c890,True
5,sources,corpus,6a25a96ec4fb66581f07c9ceac9d109c67416d1cc80ad30a6ab83039e77529ad,6a25a96ec4fb66581f07c9ceac9d109c67416d1cc80ad30a6ab83039e77529ad,True
6,comments,corpus,523eae742a6632a07eaef273307c2f9697f47f2bdf4df7b052fb940aff9b3a54,523eae742a6632a07eaef273307c2f9697f47f2bdf4df7b052fb940aff9b3a54,True
7,photometry,corpus,9d2919bbffcb45e9054206ed1c5cf7d568ea0f4a369e0492b642e750523f99f9,9d2919bbffcb45e9054206ed1c5cf7d568ea0f4a369e0492b642e750523f99f9,True
8,spectra,corpus,b55cac3b37be1af851d8a0c45e1a3fb996f851248fea61515b797fa1bd0e8fef,b55cac3b37be1af851d8a0c45e1a3fb996f851248fea61515b797fa1bd0e8fef,True
9,followup_requests,corpus,942330e7d9cff1f54871106def821411393ab93f6eeda4c37755d77393806d66,942330e7d9cff1f54871106def821411393ab93f6eeda4c37755d77393806d66,True


In [3]:
mismatched = regeneration.loc[~regeneration["match"], ["table", "stage"]].drop_duplicates()
diff_rows = []
for _, mismatch in mismatched.iterrows():
    table_name, stage = mismatch["table"], mismatch["stage"]
    on_disk_root = INTERIM_ROOT if stage == "interim" else CORPUS_ROOT
    regen_root = regen_interim if stage == "interim" else regen_corpus
    on_disk = pd.read_parquet(on_disk_root / f"{table_name}.parquet")
    regen = pd.read_parquet(regen_root / f"{table_name}.parquet")
    if list(on_disk.columns) != list(regen.columns):
        diff_rows.append({"stage": stage, "table": table_name, "row": None, "column": "<schema>",
                          "on_disk_value": list(on_disk.columns), "regenerated_value": list(regen.columns)})
        continue
    differs = on_disk.astype(str).ne(regen.astype(str))
    for row_index, col_index in zip(*differs.values.nonzero()):
        if len(diff_rows) >= 20:
            break
        diff_rows.append({"stage": stage, "table": table_name, "row": int(row_index),
                          "column": on_disk.columns[col_index],
                          "on_disk_value": on_disk.iat[row_index, col_index],
                          "regenerated_value": regen.iat[row_index, col_index]})

differences = pd.DataFrame(diff_rows, columns=["stage", "table", "row", "column",
                                               "on_disk_value", "regenerated_value"])
if differences.empty:
    print("No differences found: every regenerated table is byte-identical to the on-disk table.")
else:
    print(f"{len(differences)} differing cells shown (capped at 20).")
differences

No differences found: every regenerated table is byte-identical to the on-disk table.


,stage,table,row,column,on_disk_value,regenerated_value


In [4]:
PERSON_RE = re.compile(r"(first_name|last_name|username|contact_email|contact_phone|"
                       r"\bbio\b|allocation\.pi$|reporter|discoverer)", re.I)
FREE_TEXT_LEAVES = {"summary", "text", "comment"}


def safe(column, value):
    if PERSON_RE.search(column):
        return "<redacted>"
    if column.split(".")[-1] in FREE_TEXT_LEAVES:
        text = str(value)
        return f"{text[:20]}..." if len(text) > 20 else text
    return value


def show(row, columns):
    return {c: safe(c, row[c]) for c in columns if c in row.index}


interim_t = {n: pd.read_parquet(INTERIM_ROOT / f"{n}.parquet") for n in TABLE_NAMES}
corpus_t = {n: pd.read_parquet(CORPUS_ROOT / f"{n}.parquet") for n in TABLE_NAMES}

print("=== 1. source AT2023toh (whitespace identifier) ===")
raw_path = "data/raw/skyportal/inventory/source_inventory_grandma_base_20260720_093939/sources_page_004.json"
raw = json.load(open(ROOT / raw_path))
rec = next(s for s in raw["data"]["sources"] if s.get("id", "").strip() == "AT2023toh")
print(f"raw path: {raw_path} | raw id: {rec['id']!r}")
irow = interim_t["sources"][interim_t["sources"]["id"].astype(str).str.strip() == "AT2023toh"].iloc[0]
crow = corpus_t["sources"][corpus_t["sources"]["id"] == "AT2023toh"].iloc[0]
print("interim:", show(irow, ["id", "source_profile"]))
print("corpus: ", show(crow, ["id", "source_profiles"]))
print("decisions touched: 4 (identifier whitespace stripped)")

print("\n=== 2. source INTEGRAL-GRB231115A (multi-profile merge) ===")
for p in ["data/raw/skyportal/inventory/source_inventory_grandma_base_20260720_093939/sources_page_004.json",
         "data/raw/skyportal/inventory/source_inventory_grb_20260720_094006/sources_page_002.json"]:
    rec = next(s for s in json.load(open(ROOT / p))["data"]["sources"] if s.get("id") == "INTEGRAL-GRB231115A")
    print(f"  raw {p}: host.id={rec['host']['id'] if rec.get('host') else None}")
irows = interim_t["sources"][interim_t["sources"]["id"] == "INTEGRAL-GRB231115A"]
print("interim (profile, host.id):", list(zip(irows["source_profile"], irows["host.id"])))
crow = corpus_t["sources"][corpus_t["sources"]["id"] == "INTEGRAL-GRB231115A"].iloc[0]
print("corpus:", show(crow, ["id", "source_profiles", "host.id"]))
print("decisions touched: 1 (merged to one row), 2 (host-enrichment copy kept)")

print("\n=== 3. followup_requests id 21009 (removed capture duplicate) ===")
for d in ["EP240626A", "EP240626A-FXT"]:
    rec = next(r for r in json.load(open(ROOT / f"data/raw/skyportal/source_detail_20260724/{d}"
                                         "/followup_requests.json"))["payload"]["data"]["followup_requests"]
               if r["id"] == 21009)
    print(f"  raw dir={d}: obj_id={rec['obj_id']!r} status={rec['status']!r}")
irows = interim_t["followup_requests"][interim_t["followup_requests"]["id"] == 21009]
print("interim (source_dir, obj_id):", list(zip(irows["source_dir"], irows["obj_id"])))
crow = corpus_t["followup_requests"][corpus_t["followup_requests"]["id"] == 21009].iloc[0]
print("corpus:", show(crow, ["id", "source_dir", "obj_id", "status", "status_normalised"]))
print("decisions touched: 3 (kept source_dir == obj_id copy), 9 (status_normalised added)")

print("\n=== 4. photometry id 31672 of GRB240911A (out-of-range mjd) ===")
rec = next(r for r in json.load(open(ROOT / "data/raw/skyportal/source_detail_20260724/GRB240911A"
                                     "/photometry.json"))["payload"]["data"] if r["id"] == 31672)
print(f"raw: mjd={rec['mjd']} created_at={rec['created_at']!r}")
irow = interim_t["photometry"][interim_t["photometry"]["id"] == 31672].iloc[0]
crow = corpus_t["photometry"][corpus_t["photometry"]["id"] == 31672].iloc[0]
print("interim:", show(irow, ["id", "obj_id", "mjd", "created_at"]))
print("corpus: ", show(crow, ["id", "obj_id", "mjd", "mjd_out_of_range", "created_at"]))
print("decisions touched: 12 (mjd nulled, mjd_out_of_range flagged)")

print("\n=== 5. the single spectrum (obj_id 2026cex) ===")
raw_path = "data/raw/skyportal/source_detail_20260724/2026cex/spectra.json"
rec = json.load(open(ROOT / raw_path))["payload"]["data"]["spectra"][0]
print(f"raw path: {raw_path}")
print(f"raw: obj_id={rec['obj_id']!r} observed_at={rec['observed_at']!r} "
     f"instrument_name={rec.get('instrument_name')!r}")
irow, crow = interim_t["spectra"].iloc[0], corpus_t["spectra"].iloc[0]
print("interim:", show(irow, ["obj_id", "observed_at", "observed_at_mjd", "instrument_name"]))
print("corpus: ", show(crow, ["obj_id", "observed_at", "observed_at_mjd", "instrument_name"]))
print("decisions touched: 5 (whitespace pass, no anomaly here), 6 (empty owner.* columns "
     "dropped), 10 (created_at/modified/observed_at typed as UTC datetime)")

=== 1. source AT2023toh (whitespace identifier) ===
raw path: data/raw/skyportal/inventory/source_inventory_grandma_base_20260720_093939/sources_page_004.json | raw id: 'AT2023toh\t'
interim: {'id': 'AT2023toh\t', 'source_profile': 'grandma_base'}
corpus:  {'id': 'AT2023toh', 'source_profiles': array(['grandma_base'], dtype=object)}
decisions touched: 4 (identifier whitespace stripped)

=== 2. source INTEGRAL-GRB231115A (multi-profile merge) ===
  raw data/raw/skyportal/inventory/source_inventory_grandma_base_20260720_093939/sources_page_004.json: host.id=10194051
  raw data/raw/skyportal/inventory/source_inventory_grb_20260720_094006/sources_page_002.json: host.id=None
interim (profile, host.id): [('grandma_base', 10194051.0), ('grb', nan)]
corpus: {'id': 'INTEGRAL-GRB231115A', 'source_profiles': array(['grandma_base', 'grb'], dtype=object), 'host.id': np.float64(10194051.0)}
decisions touched: 1 (merged to one row), 2 (host-enrichment copy kept)

=== 3. followup_requests id 21009 (re

In [5]:
def count_raw_sources():
    total, ids = 0, set()
    for directory in sorted((ROOT / "data/raw/skyportal/inventory").glob("source_inventory_*_20260720_*")):
        for path in sorted(directory.glob("sources_page_*.json")):
            for record in json.load(open(path))["data"]["sources"]:
                total += 1
                ids.add(str(record.get("id", "")).strip())
    return total, len(ids)


def count_raw_detail(record_type, path_parts):
    total = 0
    for path in sorted((ROOT / "data/raw/skyportal/source_detail_20260724").glob(f"*/{record_type}.json")):
        value = json.load(open(path))
        for key in path_parts:
            value = value[key]
        total += len(value)
    return total


raw_source_records, raw_distinct_sources = count_raw_sources()
raw_counts = {
    "comments": count_raw_detail("comments", ("payload", "data")),
    "photometry": count_raw_detail("photometry", ("payload", "data")),
    "spectra": count_raw_detail("spectra", ("payload", "data", "spectra")),
    "followup_requests": count_raw_detail("followup_requests", ("payload", "data", "followup_requests")),
}

rows = [
    {"quantity": "sources (raw listing records)", "raw": raw_source_records,
     "corpus": len(corpus_t["sources"]), "expected_difference": f"{raw_source_records} -> {len(corpus_t['sources'])}",
     "explanation": "decision 1: one row per distinct source id; 182 multi-profile ids collapsed"},
    {"quantity": "sources (distinct raw ids)", "raw": raw_distinct_sources,
     "corpus": len(corpus_t["sources"]),
     "expected_difference": f"{raw_distinct_sources} -> {len(corpus_t['sources'])}",
     "explanation": "no difference: corpus rows equal the distinct raw source count"},
]
for name in ["comments", "photometry", "spectra"]:
    rows.append({"quantity": name, "raw": raw_counts[name], "corpus": len(corpus_t[name]),
                "expected_difference": f"{raw_counts[name]} -> {len(corpus_t[name])}",
                "explanation": "no row-count decision applies to this table"})
rows.append({"quantity": "followup_requests", "raw": raw_counts["followup_requests"],
            "corpus": len(corpus_t["followup_requests"]),
            "expected_difference": f"{raw_counts['followup_requests']} -> {len(corpus_t['followup_requests'])}",
            "explanation": "decision 3: 20 capture duplicates removed, kept source_dir == obj_id"})

raw_truth_counts = pd.DataFrame(rows)
raw_truth_counts

,quantity,raw,corpus,expected_difference,explanation
0,sources (raw listing records),982,800,982 -> 800,decision 1: one row per distinct source id; 182 multi-profile ids collapsed
1,sources (distinct raw ids),800,800,800 -> 800,no difference: corpus rows equal the distinct raw source count
2,comments,2950,2950,2950 -> 2950,no row-count decision applies to this table
3,photometry,7968,7968,7968 -> 7968,no row-count decision applies to this table
4,spectra,1,1,1 -> 1,no row-count decision applies to this table
5,followup_requests,2359,2339,2359 -> 2339,"decision 3: 20 capture duplicates removed, kept source_dir == obj_id"


In [6]:
span_start, span_end = pd.Timestamp("2022-11-10", tz="UTC"), pd.Timestamp("2026-07-24", tz="UTC")
cutoffs = pd.date_range(span_start, span_end, periods=5)

rows = []
for cutoff in cutoffs:
    for name in TABLE_NAMES:
        series = corpus_t[name]["created_at"]
        at_cutoff = series[series <= cutoff]
        rows.append({"T": cutoff, "table": name, "rows_at_T": len(at_cutoff), "rows_total": len(series),
                     "max_created_at_at_T": at_cutoff.max() if len(at_cutoff) else pd.NaT})

truncation = pd.DataFrame(rows)
truncation["PASS"] = truncation.apply(
    lambda r: pd.isna(r["max_created_at_at_T"]) or r["max_created_at_at_T"] <= r["T"], axis=1)
print(f"all rows PASS (max_created_at_at_T never later than T): {bool(truncation['PASS'].all())}")

probe_T = cutoffs[2]
covered = set()
for name in TABLE_NAMES:
    df = corpus_t[name]
    id_column = "id" if name == "sources" else "obj_id"
    covered |= set(df.loc[df["created_at"] <= probe_T, id_column])
covered &= set(corpus_t["sources"]["id"])
print(f"at T={probe_T}: {len(covered)} of 800 sources have any data at or before this instant")

truncation

all rows PASS (max_created_at_at_T never later than T): True
at T=2024-09-16 00:00:00+00:00: 123 of 800 sources have any data at or before this instant


,T,table,rows_at_T,rows_total,max_created_at_at_T,PASS
0,2022-11-10 00:00:00+00:00,sources,0,800,NaT,True
1,2022-11-10 00:00:00+00:00,comments,0,2950,NaT,True
2,2022-11-10 00:00:00+00:00,photometry,0,7968,NaT,True
3,2022-11-10 00:00:00+00:00,spectra,0,1,NaT,True
4,2022-11-10 00:00:00+00:00,followup_requests,0,2339,NaT,True
5,2023-10-14 00:00:00+00:00,sources,20,800,2023-09-26 07:52:09.653537+00:00,True
6,2023-10-14 00:00:00+00:00,comments,195,2950,2023-07-26 12:19:39.047310+00:00,True
7,2023-10-14 00:00:00+00:00,photometry,506,7968,2023-07-04 21:10:13.332703+00:00,True
8,2023-10-14 00:00:00+00:00,spectra,0,1,NaT,True
9,2023-10-14 00:00:00+00:00,followup_requests,355,2339,2023-09-26 07:52:09.653537+00:00,True


In [7]:
print("=== Traceability: source '2025aji' ===")
raw_path = "data/raw/skyportal/inventory/source_inventory_grandma_base_20260720_093939/sources_page_003.json"
raw_record = next(s for s in json.load(open(ROOT / raw_path))["data"]["sources"] if s["id"] == "2025aji")
print(f"raw path: {raw_path}")
print(f"raw redshift_history: {json.dumps(raw_record['redshift_history'])[:200]}...")
print(f"raw summary_history: {len(raw_record['summary_history'])} entries "
      f"(text not printed; see the notebook's personal-data rule)")

history = pd.read_parquet(CORPUS_ROOT / f"{FIELD_HISTORY_TABLE}.parquet")
rows_2025aji = history[history["source_id"] == "2025aji"]

print("\nredshift entries, raw array position vs corpus row (value, set_by_user_id):")
for index, entry in enumerate(raw_record["redshift_history"]):
    corpus_row = rows_2025aji[(rows_2025aji["field"] == "redshift")
                              & (rows_2025aji["entry_index"] == index)].iloc[0]
    print(f"  index={index}  raw=({entry['value']!r}, {entry['set_by_user_id']})  "
          f"corpus=({corpus_row['value']!r}, {corpus_row['set_by_user_id']})")

mismatches = []
for field, column, value_key in [("redshift", "redshift_history", "value"),
                                 ("summary", "summary_history", "summary")]:
    for index, entry in enumerate(raw_record[column]):
        corpus_row = rows_2025aji[(rows_2025aji["field"] == field)
                                  & (rows_2025aji["entry_index"] == index)].iloc[0]
        raw_stamp = pd.Timestamp(entry["set_at_utc"])
        raw_stamp = raw_stamp.tz_localize("UTC") if raw_stamp.tzinfo is None else raw_stamp.tz_convert("UTC")
        if (str(entry.get(value_key)) != str(corpus_row["value"])
                or raw_stamp != corpus_row["set_at_utc"]
                or entry["set_by_user_id"] != corpus_row["set_by_user_id"]):
            mismatches.append((field, index))
print(f"\nall {len(raw_record['redshift_history']) + len(raw_record['summary_history'])} raw entries "
      f"reach the corpus with value, set_at_utc and set_by_user_id intact: {not mismatches}")

print("\n=== Value reconstruction for '2025aji' at three instants ===")
instants = pd.date_range(pd.Timestamp("2023-05-01", tz="UTC"), pd.Timestamp("2026-07-24", tz="UTC"), periods=3)
for instant in instants:
    in_force = rows_2025aji[rows_2025aji["set_at_utc"] <= instant]
    redshift_now = in_force[in_force["field"] == "redshift"].sort_values("set_at_utc")
    summary_now = in_force[in_force["field"] == "summary"].sort_values("set_at_utc")
    r_value = redshift_now.iloc[-1]["value"] if len(redshift_now) else "no value yet"
    s_text = summary_now.iloc[-1]["value"] if len(summary_now) else None
    s_value = "no value yet" if s_text is None else (s_text[:40] + "..." if len(s_text) > 40 else s_text)
    at_t = history[history["set_at_utc"] <= instant]
    print(f"\nT={instant}")
    print(f"  redshift in force: {r_value}")
    print(f"  summary in force:  {s_value!r}")
    print(f"  source_field_history rows at T: {len(at_t)} of {len(history)}")
    print(f"  sources with any history entry at T: {at_t['source_id'].nunique()} of "
          f"{history['source_id'].nunique()}")

=== Traceability: source '2025aji' ===
raw path: data/raw/skyportal/inventory/source_inventory_grandma_base_20260720_093939/sources_page_003.json
raw redshift_history: [{"value": "2.15", "set_at_utc": "2025-01-29T13:29:00.561728", "uncertainty": null, "set_by_user_id": 113}, {"value": "2.151", "origin": "GCN 39071, 39073", "set_at_utc": "2025-06-06T23:26:40.672783",...
raw summary_history: 8 entries (text not printed; see the notebook's personal-data rule)

redshift entries, raw array position vs corpus row (value, set_by_user_id):
  index=0  raw=('2.15', 113)  corpus=('2.15', 113)
  index=1  raw=('2.151', 92)  corpus=('2.151', 92)
  index=2  raw=('2.15', 92)  corpus=('2.15', 92)
  index=3  raw=('2.151', 92)  corpus=('2.151', 92)

all 12 raw entries reach the corpus with value, set_at_utc and set_by_user_id intact: True

=== Value reconstruction for '2025aji' at three instants ===

T=2023-05-01 00:00:00+00:00
  redshift in force: no value yet
  summary in force:  'no value yet'
  sourc

In [8]:
final_rows = []
for stage, root in [("interim", INTERIM_ROOT), ("corpus", CORPUS_ROOT)]:
    for name in TABLE_NAMES:
        final_rows.append({"stage": stage, "table": name, "sha256": sha256_of_file(root / f"{name}.parquet")})
final_rows.append({"stage": "corpus", "table": FIELD_HISTORY_TABLE,
                   "sha256": sha256_of_file(CORPUS_ROOT / f"{FIELD_HISTORY_TABLE}.parquet")})
final_hashes = pd.DataFrame(final_rows)

baseline_hashes = baseline[["file", "sha256"]].copy()
baseline_hashes["stage"] = baseline_hashes["file"].str.split("/").str[0]
baseline_hashes["table"] = baseline_hashes["file"].str.split("/").str[1].str.replace(".parquet", "", regex=False)
merged = final_hashes.merge(baseline_hashes, on=["stage", "table"], suffixes=("_now", "_baseline"))
unchanged = bool((merged["sha256_now"] == merged["sha256_baseline"]).all())
print(f"interim and corpus files unchanged from the cell-2 baseline: {unchanged}")
print(f"files compared: {len(merged)}")
print(merged[["stage", "table", "sha256_now", "sha256_baseline"]].to_string(index=False))

shutil.rmtree(tmp_root)
print(f"\ntemporary directory removed: {not tmp_root.exists()}")
print(f"path was: {tmp_root}")

interim and corpus files unchanged from the cell-2 baseline: True
files compared: 11
  stage                table                                                       sha256_now                                                  sha256_baseline
interim              sources f50ef622a28285f750d940fc5dfb39d5f7405360c0519f63d6600bde49e49a6b f50ef622a28285f750d940fc5dfb39d5f7405360c0519f63d6600bde49e49a6b
interim             comments 4dc1e66febd156516d49867b20a6a1a55363c1334ba1bd3544a435974c7774b0 4dc1e66febd156516d49867b20a6a1a55363c1334ba1bd3544a435974c7774b0
interim           photometry db93bd722c22d938df389b6cdf2815d4d5c58b4b38f8da54af3085df8b4baa0f db93bd722c22d938df389b6cdf2815d4d5c58b4b38f8da54af3085df8b4baa0f
interim              spectra ddb9ce9421dfa87c3990398d1aac078393d83bf7a385d6edbbd89be3290a69f3 ddb9ce9421dfa87c3990398d1aac078393d83bf7a385d6edbbd89be3290a69f3
interim    followup_requests 22d695fb99e75e44e23ed69daa5f436de908e9402ebf7ea80794f2d03c38c890 22d695fb99e75e44e23ed69daa

## What this test establishes

The corpus is a deterministic function of the raw captures. Running the
two scripts in a clean directory reproduces all eleven files byte for byte,
at both stages. Regenerating it requires nothing that is not in this
repository and in `data/raw/`.

Five records were followed from raw JSON to their final row, each
crossing a different decision: an identifier cleaned, a multi-profile
merge, a capture duplicate removed, an impossible MJD nulled, and
timestamps typed. Counts taken directly from the raw JSON match the
corpus, and every row-count difference is attributed to the decision
responsible for it.

Causal truncation was verified at five instants spread across the span:
across all 25 checks, no surviving row carries a `created_at` later than
its cut-off. At 2024-09-16, 123 of the 800 sources had any data at all.

Scope of that truncation: event records — comments, photometry, spectra
and follow-up requests — are immutable once created, and cutting them by
`created_at` is exact. Rows in `sources` are records that get updated, so
a cut determines which sources existed at instant T while their values
are those held at capture time. For `redshift` and `summary` this limit
does not apply: `source_field_history` carries 1,357 dated changes across
266 sources, and the value in force at any instant is the latest entry
with `set_at_utc` at or before it. No other field of `sources` carries a
change history.